In [ ]:
# 대한민국 구석구석 사이트 상세 정보 수집하기-2026_03_05일 수정함
#Step 1. 필요한 모듈을 로딩합니다
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time          
import pandas as pd    
import math
import pyautogui
import os

#Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
query_txt = input('1.정보를 수집할 키워드는 무엇입니까?: ')

cnt = int(input('2.몇 건의 정보를 수집할까요? :'))
page_cnt = math.ceil( cnt / 8 )
print('총 %s 건의 정보를 수집하기 위해 %s 페이지까지 이동할 예정입니다' %(cnt , page_cnt))

#Step 3. 수집된 데이터를 저장할 폴더 이름 입력받기 
f_dir = input("3.파일을 저장할 폴더명만 쓰세요(기본값:c:\\py_temp\\):")
if f_dir == '' :
    f_dir="c:\\py_temp\\"
    
print("\n")    

#Step 4. 검색어 입력한 후 검색하여 해당 장르로 이동하기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

query_url = 'https://korean.visitkorea.or.kr/'

driver.get(query_url)
time.sleep(2)
driver.maximize_window()

#Step 4. 자동으로 검색어 입력 후 조회하기
driver.find_element(By.XPATH,'//*[@id="headerSearchBtn"]').click()
time.sleep(1)
element = driver.find_element(By.ID,'inp_search_index')
element.send_keys(query_txt)
element.send_keys("\n")

# 위치기반 서비스 물어보면 동의
try :
    driver.find_element(By.XPATH,'//*[@id="locationServicePop"]/div[1]/div/div[2]/a[2]').click()
except :
    pass
  

# 여행지 탭 클릭
driver.find_element(By.XPATH,'//*[@id="swiper-wrapper-4c0d2bb4b1059bb109"]/div[3]/button').click()
time.sleep(2)                    

# Step 5. 본문 내역 추출하기
no2=[]           # 게시글 번호 컬럼
title2=[ ]       # 게시글 제목 컬럼
org2=[]          # 지자체명 컬럼
contents2=[ ]     # 본문 내용 컬럼
tag2=[ ]         # 해시태그 컬럼

no = 1           # 게시글 번호 초기값

# 마우스 휠을 자동으로 돌리는 사용자정의함수
def scroll_down(driver):      
    driver.execute_script("window.scrollBy(0,170);")
    time.sleep(1)

def scroll_up(driver):      
    driver.execute_script("window.scrollBy(0,-1500);")
    time.sleep(1)


for a in range(1,page_cnt+1) :
    print('')
    time.sleep(5)
    print('현재 %s 페이지에 있는 정보를 수집합니다' %a)

    if a >= 2 :
        scroll_up(driver)
        
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    content_list = soup.find('div','spot_list').find_all('li')
   
    cnt_no = 1
    
    for b in content_list:
               
        try :
            title = b.find('div','tit_wrap').get_text().strip()
        except :
            continue
        else :
            #scroll_down(driver)

            print('-' *100)
            
            # 1. 게시글 번호
            print("\n")
            print("%s 번째 정보를 추출하고 있습니다============" %no)
            no2.append(no)                         
            print("1.번호 : %s" %no)

            title2.append(title)
            print("2.관광지명 : %s" %title)
            
            # 3. 지자체이름
            org_all = b.select('div.area > span')
            try:
                org = org_all[0].get_text()
            except :
                org = b.find('span','area').get_text()  
            org2.append(org)
            print("3.지자체이름: %s" %org)
                   
            #4. 해시태그
            try :
                tag = b.find('div','tag').get_text().replace("#"," #")
            except :
                tag = ' '    
            tag2.append(tag)
            print("4.해시태그: %s" %tag)
                       
            #5. 본문   
            driver.find_element(By.XPATH,'//*[@id="contents"]/div/div[2]/div[2]/div[3]/div/ul/li[%s]/a' %cnt_no).click()

            cnt_no += 1
            
            time.sleep(5)
            
            html_2 = driver.page_source
            soup_2 = BeautifulSoup(html_2, 'html.parser')
            
            content1 = soup_2.find('div','inr_wrap')
            # content2 = soup_2.find('div','box_txtPhoto')
            # content3 = soup_2.find('div','titleType1')
            
            if content1 :                
                content = content1.find('div','inr').find('p').get_text()
                contents2.append(content)
                print('5.본문내용:',content)
                
            # elif content2 :
            #     content_1 = soup_2.find('div','box_txtPhoto')
            #     try :
            #         content_1.style.decompose()
            #         content_1.script.decompose()
            #     except :
            #         content_2 = content_1.find_all('div','txt_p')

            #         for b in content_2 :
            #             content = b.get_text().strip()
            #             contents2.append(content)
            #             print('5.본문내용:',content)
            #     else :
            #         content_2 = content_1.find_all('div','txt_p')
            #         for b in content_2 :
            #             content = b.get_text().strip()
            #             contents2.append(content)
            #             print('5.본문내용:',content)
            # elif content3:
            #     title = content1.select('#topTitle').get_text()
            #     title2.append(title)
            
            driver.back()
            
            time.sleep(3)
                       
            no += 1         # 전체 게시글 번호용 값
            
            if no > cnt :
                 break
                
            time.sleep(2)
            
    a += 1
    try :
        driver.find_element(By.LINK_TEXT,'%s' %a).click()
    except :
        driver.find_element(By.LINK_TEXT,'다음').click()

    time.sleep(5)

# Step 6. 결과 저장하기
n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' %(n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

os.makedirs(f_dir+'대한민국구석구석'+'-'+s+'-'+query_txt)

fc_name = f_dir+'대한민국구석구석'+'-'+s+'-'+query_txt+'\\'+'대한민국구석구석'+'-'+s+'-'+query_txt+'.csv'
fx_name = f_dir+'대한민국구석구석'+'-'+s+'-'+query_txt+'\\'+'대한민국구석구석'+'-'+s+'-'+query_txt+'.xlsx'

df = pd.DataFrame()
df['번호']=no2
df['제목']=pd.Series(title2)
df['지자체명']=pd.Series(org2)
df['본문내용']=pd.Series(contents2)
df['해시태그']=pd.Series(tag2)

# xlsx , csv 형태로 저장하기
df.to_excel(fx_name,index=False, engine='openpyxl')
df.to_csv(fc_name,index=False, encoding="utf-8-sig")

print("\n") 
print("=" *80)
print("1.크롤링을 요청한 총 %s 건 중에서 %s 건의 데이터를 수집 완료 했습니다" %(cnt,no-1))
print('2.xlsx 저장경로: ',fx_name)
print('3.csv 저장경로: ',fc_name)
print("=" *80)
